# FHIR Data Analytics with Synthea

Parsing FHIR R4 patient bundles and generating population health analytics.

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from parser import parse_bundle
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

## 1. Parse Sample FHIR Bundle

In [ ]:
with open('../data/sample/sample_bundle.json') as f:
    bundle = json.load(f)
records = parse_bundle(bundle)
for rtype, recs in records.items():
    if recs:
        print(f'{rtype}: {len(recs)} record(s)')
        print(pd.DataFrame(recs).to_string())
        print()

## 2. Simulated Population Analytics (500 patients)

In [ ]:
# Generate synthetic population
N = 500
ages = np.concatenate([
    np.random.randint(0, 18, 75),
    np.random.randint(18, 65, 275),
    np.random.randint(65, 95, 150)
])
conditions = [
    ('Hypertension', 0.30), ('Type 2 Diabetes', 0.15), ('Hyperlipidemia', 0.28),
    ('Obesity', 0.22), ('Anxiety', 0.18), ('Depression', 0.16),
    ('Asthma', 0.12), ('COPD', 0.08), ('Coronary Artery Disease', 0.09),
    ('Osteoarthritis', 0.20)
]

cond_counts = {c: int(N * p) for c, p in conditions}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(ages, bins=20, color='#1E88E5', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].set_title('Synthetic Patient Age Distribution (N=500)')
axes[0].axvline(np.median(ages), color='red', linestyle='--', label=f'Median: {np.median(ages):.0f}')
axes[0].legend()

cond_names = [c for c, _ in conditions]
cond_pct = [p * 100 for _, p in conditions]
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(conditions)))
bars = axes[1].barh(cond_names[::-1], cond_pct[::-1], color=colors, alpha=0.85)
axes[1].set_xlabel('Prevalence (%)')
axes[1].set_title('Condition Prevalence in Synthea Population')
axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/population_overview.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. FHIR Resource Structure

Key FHIR R4 resources used in this pipeline:

| Resource | Purpose |
|----------|---------|
| `Patient` | Demographics, identifiers |
| `Condition` | Diagnoses (SNOMED CT coded) |
| `MedicationRequest` | Prescriptions (RxNorm coded) |
| `Observation` | Lab results (LOINC coded) |
| `Encounter` | Visit records |

All resources linked by `patient.id` via the `subject.reference` field.

In [ ]:
print('Sample parsed patient record:')
print(json.dumps(records['Patient'][0], indent=2))
print('\nSample parsed condition record:')
print(json.dumps(records['Condition'][0], indent=2))